In [1]:
from data.preprocessing.pipeline import Pipeline

In [2]:
pipeline = Pipeline(download=False)

Extracting raw features: 100%|██████████| 888/888 [00:09<00:00, 90.63it/s]


In [7]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

# ---------------------------
# 1. Извлечение данных
# ---------------------------
train_df, test_df = pipeline.get_processed_data()

# 2. Целевая переменная и признаки
y = train_df["taste_cluster"]
feature_cols = [col for col in train_df.columns if col.startswith("feat_")]
X = train_df[feature_cols].values

# 3. Разделение на train / val (test_df не используется для анализа ошибок)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Сохраняем Pubchem_ID для валидационной части (для анализа)
val_ids = train_df.iloc[X_val.shape[0]:]  # ❌ ОШИБКА! Нельзя так делать.

# Правильный способ: использовать индексы из train_test_split
# Для этого разбиваем не только X и y, но и Pubchem_ID
train_ids = train_df["Pubchem_ID"].values
X_train, X_val, y_train, y_val, train_ids_split, val_ids = train_test_split(
    X, y, train_ids, test_size=0.2, stratify=y, random_state=42
)

# 4. Обучение модели
model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    num_class=len(np.unique(y)),
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

# 5. Предсказания на валидации
y_val_pred = model.predict(X_val)

# 6. Метрики
val_f1_macro = f1_score(y_val, y_val_pred, average='macro')
print(f"Validation F1-macro: {val_f1_macro:.4f}")

print("\n=== Classification Report (Validation Set) ===")
print(classification_report(y_val, y_val_pred))

# 7. Анализ ошибок на валидации
errors_mask = y_val_pred != y_val
num_errors = errors_mask.sum()
print(f"\nОшибок на валидации: {num_errors} из {len(y_val)}")

if num_errors > 0:
    # Собираем информацию об ошибках
    errors_df = pd.DataFrame({
        "Pubchem_ID": val_ids[errors_mask],
        "true_label": y_val[errors_mask],
        "predicted_label": y_val_pred[errors_mask]
    })
    
    # Добавляем уверенность модели (макс. вероятность)
    y_val_proba = model.predict_proba(X_val)[errors_mask]
    errors_df["max_proba"] = y_val_proba.max(axis=1)
    
    print("\nПримеры ошибок на валидации (первые 10):")
    print(errors_df.head(10))
    
    # Опционально: сохранить в файл
    # errors_df.to_csv("val_errors.csv", index=False)
else:
    print("Модель не ошиблась ни разу на валидации!")

Validation F1-macro: 0.4238

=== Classification Report (Validation Set) ===
              precision    recall  f1-score   support

           0       0.48      0.63      0.55       106
           1       0.41      0.34      0.37        59
           2       0.53      0.48      0.50        44
           3       0.42      0.42      0.42        81
           4       0.58      0.56      0.57        57
           5       0.40      0.22      0.29         9
           6       0.43      0.53      0.47        17
           7       0.30      0.17      0.22        40

    accuracy                           0.46       413
   macro avg       0.44      0.42      0.42       413
weighted avg       0.46      0.46      0.45       413


Ошибок на валидации: 221 из 413

Примеры ошибок на валидации (первые 10):
      Pubchem_ID  true_label  predicted_label  max_proba
204        10883           4                3   0.569746
1649       15619           3                0   0.518060
1828       24114           